# Assignment Instructions - Part 3: AutoML as Evolutionary Pipeline Search

In analytical modeling, genetic search and StackGP searched for **equations** that explained data. Here, TPOT searches for a fully data-driven predictive **workflow**: preprocessing steps, feature transformations, model family, and hyperparameters. The goal is not to let software choose science. The goal is to make the search visible, bounded, reproducible, and open to criticism.

This activity uses **CASH**: *combined algorithm selection and hyperparameter optimization*. You will let TPOT run a deliberately small CASH search, compare it with the baseline from Part 2, and decide whether its recommendation deserves trust.

## Learning goals

By the end of this activity, you should be able to:

- explain how TPOT uses evolutionary search to address a bounded CASH problem;
- run an AutoML search without leaking the final test data into its decisions;
- inspect and save the pipeline that AutoML selected; and
- identify why a high AutoML score does not settle a scientific question.

You are expected to read the [TPOT documentation](https://epistasislab.github.io/tpot/) and the [scikit-learn model-selection guide](https://scikit-learn.org/stable/model_selection.html). This notebook is a working scaffold, not an API reference.

## 1. What exactly is TPOT optimizing?

A conventional model choice might be: “fit logistic regression with these settings.” A **pipeline** adds choices before and around that estimator: imputation, scaling, feature selection, transformations, and the estimator itself.

CASH searches both:

1. **Algorithm selection:** Which pipeline structure and estimator family should be tried?
2. **Hyperparameter optimization:** Which settings should each candidate use?

TPOT uses evolutionary (genetic) programming to generate, evaluate, vary, and retain pipeline candidates. That connection is real, but the objects being evolved have changed:

| Earlier analytical-modeling activity | This activity |
|---|---|
| Candidate equation | Candidate scikit-learn pipeline |
| Fit to observed values | Cross-validation score |
| Interpretable mathematical form | Predictive workflow and its assumptions |
| Search may discover a relation | Search may discover a high-scoring pipeline |

Neither search establishes causation, validates a bad dataset, or replaces domain expertise. In particular, TPOT is **AutoML**, not necessarily “meta-learning” in the narrower research sense of learning across many prior tasks.

In [ ]:
import sys

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import tpot

from sklearn.datasets import load_breast_cancer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, balanced_accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from tpot import TPOTClassifier

RANDOM_SEED = 802

print(f"Python: {sys.version.split()[0]}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"TPOT: {tpot.__version__}")

## 2. Define the question *before* searching

We use the breast-cancer diagnostic dataset bundled with scikit-learn. The target is whether a tumor is classified as malignant or benign from 30 numeric measurements. The dataset is convenient for a classroom activity; it is **not** a basis for clinical decisions.

Part 2's rule-of-10 check applies before AutoML: this dataset has 569 observations and 30 features, or about 19 observations per feature. That clears the rule of 10 as a first-pass screen, not as a guarantee. Check class balance and feature types before you search.

In [ ]:
dataset = load_breast_cancer(as_frame=True)
X = dataset.data
y = dataset.target

print(f"Observations: {len(X)}")
print(f"Features: {X.shape[1]}")
print(f"Observations per feature: {len(X) / X.shape[1]:.1f}")
print("\nClass counts:")
print(y.value_counts().rename(index=dict(enumerate(dataset.target_names))))
X.head()

### Discuss before continuing

- Which target class would be more costly to misclassify in the real diagnostic problem? Is raw accuracy an adequate metric?
- This dataset is already complete and numeric. What preprocessing would you need for your own data, and what must be learned only from the training folds?
- Is the rule of 10 meaningful for every model that TPOT might try? Search for *events per variable* and summarize one caveat.

## 3. Protect a final test set and establish a baseline

The test set is a sealed envelope. TPOT will not see it while generating or ranking pipelines. We will open it only after selecting a pipeline.

We use **balanced accuracy** because a classifier that performs well only on the majority class can look deceptively good under ordinary accuracy. Read the scikit-learn documentation for its definition.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_SEED,
)

baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
hand_built = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2_000, random_state=RANDOM_SEED),
).fit(X_train, y_train)

for name, model in [("Dummy baseline", baseline), ("Part 2 logistic regression", hand_built)]:
    score = balanced_accuracy_score(y_test, model.predict(X_test))
    print(f"{name:28s} held-out balanced accuracy: {score:.3f}")

## 4. Run a bounded CASH search

The limits below are part of the experiment design, not technical clutter:

- `search_space="linear"` constrains which pipeline structures TPOT may evolve.
- `cv=5` means candidates are evaluated with five-fold cross-validation on **training data only**.
- `max_time_mins` and `max_eval_time_mins` bound the cost and prevent one candidate from consuming the session.
- `random_state` makes this particular stochastic run more reproducible.
- `n_jobs=1` is conservative and portable for a laptop. A carefully documented HPCC run may use more resources.

This is intentionally a short classroom search. A longer run changes the search budget and increases the risk of selecting a lucky candidate. Do not increase its limits until you can state what new evidence you expect and how you will validate it.

In [ ]:
SEARCH_MINUTES = 1
MAX_CANDIDATE_MINUTES = 0.2

tpot_search = TPOTClassifier(
    search_space="linear",
    scorers=["balanced_accuracy"],
    cv=5,
    max_time_mins=SEARCH_MINUTES,
    max_eval_time_mins=MAX_CANDIDATE_MINUTES,
    n_jobs=1,
    random_state=RANDOM_SEED,
    verbose=2,
)
tpot_search.fit(X_train, y_train)

## 5. Inspect the recommendation before trusting it

The selected object is ordinary scikit-learn code, not a black box. Read the pipeline. Identify every transformer and the final estimator, then look up any unfamiliar step in the scikit-learn documentation.

`evaluated_individuals` is a record of candidates searched during this run. It is **not** an unbiased leaderboard: all candidates were repeatedly considered while the search evolved. The cross-validation score is useful for choosing among training-set candidates, but it is not the final estimate of real-world performance.

In [ ]:
selected_pipeline = tpot_search.fitted_pipeline_
print(selected_pipeline)

results = tpot_search.evaluated_individuals.copy()
results.sort_values(
    by="balanced_accuracy_score",
    ascending=False,
)[["balanced_accuracy_score", "Generation", "Pareto_Front", "Individual"]].head(10)

In [ ]:
auto_predictions = tpot_search.predict(X_test)
auto_score = balanced_accuracy_score(y_test, auto_predictions)

print(f"AutoML held-out balanced accuracy: {auto_score:.3f}")
ConfusionMatrixDisplay.from_predictions(
    y_test,
    auto_predictions,
    display_labels=dataset.target_names,
    cmap="Blues",
)
plt.title("TPOT pipeline on the held-out test set")
plt.show()

### Investigate

1. Compare the three held-out scores: dummy baseline, your hand-built logistic-regression pipeline, and TPOT. Is the difference practically meaningful for the problem?
2. Does the selected pipeline use feature selection or dimension reduction? Relate that choice to the curse of dimensionality discussion from Part 2. Does TPOT's choice prove that the removed features are scientifically irrelevant?
3. Inspect the confusion matrix. Which error type would be more serious in a real diagnostic setting? Change the scorer only after you can justify the metric in writing.
4. Rerun the search with a different `RANDOM_SEED` once. Does TPOT select the same structure? If not, what does that say about your confidence in a one-minute AutoML result?
5. In the old TPOT tutorial, an open-ended search was presented as a route to the “best” pipeline. Explain why “best” must include a metric, a search budget, a validation protocol, and an intended use.

## 6. Handoff: save the pipeline and its experiment record

TPOT 1.1 exposes the chosen pipeline through `fitted_pipeline_`; unlike the older tutorial API, it does not use `tpot.export(...)`. Save the fitted pipeline **and** enough context for a teammate to interpret it. A saved model without the data definition, split strategy, metric, dependency versions, and limitations is not a reproducible result.

Run the following cell only after you have decided the result is worth preserving. The model file is a binary artifact; do not load an untrusted model file from someone else.

In [ ]:
MODEL_PATH = "tpot_breast_cancer_pipeline.joblib"
EXPERIMENT_PATH = "tpot_breast_cancer_experiment.csv"

joblib.dump(selected_pipeline, MODEL_PATH)
results.to_csv(EXPERIMENT_PATH, index=False)

print(f"Saved pipeline: {MODEL_PATH}")
print(f"Saved candidate record: {EXPERIMENT_PATH}")
print("Before sharing, record the data source, target definition, split seed,")
print("scoring metric, search limits, package versions, and known limitations.")

## 7. Apply this workflow to your own data (planning exercise)

Do **not** paste protected or sensitive research data into a shared notebook. First write a plan:

1. State one prediction question, target, unit of observation, and intended decision.
2. Count observations, features, target classes/events, missing values, and protected-data constraints.
3. Choose a baseline, metric, train/test strategy, and a time budget before running TPOT.
4. Decide how preprocessing will live inside the cross-validation pipeline. For mixed numeric/categorical data, investigate `ColumnTransformer`.
5. State the one external or held-out evaluation that could challenge the chosen pipeline.

If this plan is weak, AutoML makes it easier to produce a polished but untrustworthy number. Fix the experimental design first.

## Reflection

In a short response, address all three prompts:

- How is TPOT's evolutionary pipeline search similar to, and different from, the symbolic-regression/StackGP search used in analytical modeling?
- What evidence from this notebook supports or limits trust in the selected pipeline? Explicitly distinguish cross-validation evidence from held-out-test evidence.
- For a dataset from your own research, what would make CASH useful, and what must remain a human decision?

## Before the next class

Keep your notebook, the saved pipeline only if you intend to preserve it, and a short experiment record. Be prepared to explain one choice TPOT made and one choice that it could not responsibly make for you.